# Statistical Methods in Imaging (SMI) Conference, 2026.
# Empowering Large Language Models with Statistics: A Practical Tutorial for Medical Imaging
**Ernest (Khashayar) Namdar, Dominik A. Deniffel, Pascal Tyrrell**

This tutorial focuses on classifying Acute Ischemic Stroke (AIS) from Radiology reports. We will use the `Label` and `Text` columns for a probabilistic classification task.

Several similar pipelines were discussed in our publication:
```bibtex
@inproceedings{10.1117/12.3084682,
author = {Khashayar Namdar and Saeidehsadat Mirjalili and Lauren Erdman and Dominik A. Deniffel and Keith Brunt and Leo Anthony Celi},
title = {{Comparative evaluation of machine learning and large language model pipelines for identifying acute ischemic stroke in radiology reports}},
volume = {13926},
booktitle = {Medical Imaging 2026: Computer-Aided Diagnosis},
editor = {Axel Wism{\"u}ller and Thomas Martin Deserno},
organization = {International Society for Optics and Photonics},
publisher = {SPIE},
pages = {139261S},
keywords = {Stroke, NLP, Machine Learning, Large Language Models},
year = {2026},
doi = {10.1117/12.3084682},
URL = {https://doi.org/10.1117/12.3084682}
}
```

Also, the source for the dataset is:
```bibtex
@article{10.1371/journal.pone.0212778,
    doi = {10.1371/journal.pone.0212778},
    author = {Kim, Chulho AND Zhu, Vivienne AND Obeid, Jihad AND Lenert, Leslie},
    journal = {PLOS ONE},
    publisher = {Public Library of Science},
    title = {Natural language processing and machine learning algorithm to identify brain MRI reports with acute ischemic stroke},
    year = {2019},
    month = {02},
    volume = {14},
    url = {https://doi.org/10.1371/journal.pone.0212778},
    pages = {1-13},
    number = {2},
}
```


## Probabilistic AIS Classification with Decoder LLMs
In this section, we extract raw logit scores from the language model for the words "yes" and "no" to calculate a continuous probability instead of returning a hard binary inference. We then use 10 folds (treated as bootstrap repeats) to calculate the Mean AUC and its 95% Confidence Interval.

*Note: The cross-validation fold structure is used here merely to calculate confidence intervals for fair statistical comparison with the Encoder methods. The Decoder LLM itself is zero-shot and is not trained on these folds.*

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
import math
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

torch.random.manual_seed(0)

model_name = "microsoft/MediPhi"
# Initialize the Large Language Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",          # Automatically loads the model weights onto the GPU for faster inference
    torch_dtype="auto",         # Uses the optimal precision (e.g., float16) to save GPU memory
    trust_remote_code=True,     # Required for some newer HuggingFace models with custom architectures
)
# The tokenizer converts raw text into numerical tokens the model can understand
tokenizer = AutoTokenizer.from_pretrained(model_name)

def build_prompt(text):
    return f"""You are a medical AI assistant trained to classify radiology MRI reports.
Determine whether there is evidence of **acute ischemic stroke** in the following report.
Reply with a single word only: "yes" or "no".

Report:
---
{text}
---

Answer:"""

# Pre-calculate target token IDs
# Models might generate token with or without leading space or capitalization
yes_ids = [
    tokenizer.encode("yes", add_special_tokens=False)[-1],
    tokenizer.encode("Yes", add_special_tokens=False)[-1],
    tokenizer.encode(" yes", add_special_tokens=False)[-1],
    tokenizer.encode(" Yes", add_special_tokens=False)[-1]
]
no_ids = [
    tokenizer.encode("no", add_special_tokens=False)[-1],
    tokenizer.encode("No", add_special_tokens=False)[-1],
    tokenizer.encode(" no", add_special_tokens=False)[-1],
    tokenizer.encode(" No", add_special_tokens=False)[-1]
]


In [ ]:
# Load Data
df = pd.read_csv('../data/AIS.csv')

# Chop into 10 subtasks to prevent memory bloat
# Chop the dataset into smaller subtasks to prevent memory bloat over thousands of iterations
num_chunks = 10
df_chunks = np.array_split(df, num_chunks)

all_probs = []
all_ids = []

for i, chunk in enumerate(df_chunks):
    print(f"Processing subtask {i+1}/{num_chunks}...")
    chunk_probs = []
    chunk_ids = []
    
    for index, row in tqdm(chunk.iterrows(), total=len(chunk)):
        text = row['Text']
        prompt = build_prompt(text)
        messages = [
            {"role": "system", "content": "You are a radiologist."},
            {"role": "user", "content": prompt},
        ]
        
        # Format messages into model input
        # Format messages into model input
        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
            input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            # Fallback if no chat template is found
            input_text = f"System: {messages[0]['content']}\nUser: {messages[1]['content']}\nAssistant:"
            
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
        
        # Extract logits for the next token without calculating gradients (saves memory)
        with torch.no_grad():
            outputs = model(**inputs)
            # We grab the logits for the very last token in the sequence (index -1)
            next_token_logits = outputs.logits[0, -1, :]
        
        # Find max logit among the 'yes' variants and 'no' variants
        yes_logit = max([next_token_logits[tid].item() for tid in yes_ids])
        no_logit = max([next_token_logits[tid].item() for tid in no_ids])
        
        # Softmax over just 'yes' and 'no' variants to get a continuous probability between 0.0 and 1.0
        # Subtracting max_logit is a standard trick for numerical stability to prevent overflow
        max_logit = max(yes_logit, no_logit)
        exp_yes = math.exp(yes_logit - max_logit)
        exp_no = math.exp(no_logit - max_logit)
        prob_yes = exp_yes / (exp_yes + exp_no)
        
        chunk_probs.append(prob_yes)
        chunk_ids.append(row['ID'])
        
    all_probs.extend(chunk_probs)
    all_ids.extend(chunk_ids)
    
    # Aggregating intermediate results to disk
    temp_df = pd.DataFrame({'Id': chunk_ids, 'Prob_Yes': chunk_probs})
    temp_df.to_csv(f'../results/LM_prob_inference_chunk_{i+1}.csv', index=False)
    
    # Clear Python garbage and PyTorch GPU cache to sustain inference speed and prevent OOM errors
    gc.collect()
    torch.cuda.empty_cache()

# Save aggregated inferences to CSV
results_df = pd.DataFrame({'Id': all_ids, 'Prob_Yes': all_probs})
results_df.to_csv('../results/LM_prob_inference.csv', index=False)
print("Aggregated probabilities saved to ../results/LM_prob_inference.csv")


In [5]:
# Evaluate Probabilistic Performance (AUC & 95% CI)
# Load folds from Part1
try:
    folds_df = pd.read_csv('../../Part1_Encoder_LLM_for_Classification/data/Folds.csv')
except FileNotFoundError:
    print("Error: Folds.csv not found in Part1 directory. Please check the path.")
    folds_df = None

if folds_df is not None:
    # Merge predictions with fold assignments and ground truth labels
    merged_df = results_df.merge(folds_df, left_on='Id', right_on='ID')
    merged_df = merged_df.merge(df[['ID', 'Label']], on='ID')
    
    aucs = []
    for fold in sorted(merged_df['Fold'].unique()):
        fold_data = merged_df[merged_df['Fold'] == fold]
        y_true = fold_data['Label']
        y_prob = fold_data['Prob_Yes']
        
        # Only calculate AUC if both classes are present in the fold
        if len(y_true.unique()) > 1:
            auc = roc_auc_score(y_true, y_prob)
            aucs.append(auc)
    
    if aucs:
        mean_auc = np.mean(aucs)
        std_auc = np.std(aucs)
        # Calculate 95% Confidence Interval for 10 repeats
        ci = 1.96 * std_auc / np.sqrt(len(aucs))
        
        print(f"Results across {len(aucs)} folds:")
        print(f"Mean AUC: {mean_auc:.4f}")
        print(f"95% CI:   ± {ci:.4f}")
        print(f"AUC Range: [{mean_auc - ci:.4f}, {mean_auc + ci:.4f}]")
    else:
        print("Not enough data in folds to calculate AUC.")


Results across 10 folds:
Mean AUC: 0.9695
95% CI:   ± 0.0052
AUC Range: [0.9642, 0.9747]
